In [15]:

from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.7
)



class LLMState(TypedDict):

    question: str
    answer: str

def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state




In [16]:
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [17]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

The Moon is on average about **384 400 kilometers** (≈ 238 900 miles) from Earth.  
Its distance varies because its orbit is slightly elliptical:

- **Perigee (closest point)** ≈ 363 300 km (≈ 225 700 mi)  
- **Apogee (farthest point)** ≈ 405 500 km (≈ 251 900 mi)

So, depending on where the Moon is in its orbit, the distance ranges roughly from 363 000 km to 406 000 km, with the mean value around 384 400 km.


In [18]:
model.invoke('How far is moon from the earth?').content

'The Moon is, on average, about **384\u202f400\u202fkilometres (≈\u202f238\u202f900\u202fmiles)** from the Earth.  \n\n- **Perigee (closest point):** ~363\u202f300\u202fkm  \n- **Apogee (farthest point):** ~405\u202f500\u202fkm  \n\nThe distance varies because the Moon’s orbit is elliptical. Light takes roughly **1.28\u202fseconds** to travel from the Moon to Earth.'